# NiyamTrace-X Q1 Experiment 4 — Statistical Validity, Dataset Audit, and Reproducibility Pack

**Goal:** turn the current paper counts and public benchmark into an auditable statistical
package.

It recomputes exact confidence intervals and paired tests, audits the public 10,000-row
dataset for duplication/template leakage, performs zero-event power analysis, and creates a
fail-loud reproducibility report.

If exact per-case frozen files are placed in `/content/paper_raw/`, the notebook additionally
runs group-clustered bootstrap confidence intervals.

In [ ]:
# Reproducible setup: pin the exact public repository commit audited in the manuscript.
REPO_URL = "https://github.com/bnssaanirudh/NiyamTrace-X.git"
PINNED_COMMIT = "c14661dbd11c42ebd1019b6a1a5c49b8643da137"

!rm -rf /content/NiyamTrace-X
!git clone -q $REPO_URL /content/NiyamTrace-X
%cd /content/NiyamTrace-X
!git checkout -q $PINNED_COMMIT

!pip -q install -e /content/NiyamTrace-X/niyamtrace
!pip -q install pandas numpy scipy scikit-learn matplotlib tqdm statsmodels nbformat

from pathlib import Path
import os, json, math, random, hashlib, statistics, itertools, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path("/content/NiyamTrace-X")
NIYAM = ROOT / "niyamtrace"
RESULTS = Path("/content/niyamtrace_q1_results")
RESULTS.mkdir(exist_ok=True)
print("Pinned commit:", PINNED_COMMIT)
print("Results:", RESULTS)

In [ ]:
from scipy.stats import beta, binomtest
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import re

PAPER={
    "Qwen3.5-122B":{
        "n":2000,"correct":1997,"semantic_correct":1705,"non_allow":1496,
        "unsafe_allow":0,"false_blocks":0,"raw_correct":1891,"raw_unsafe_allow":80,
        "raw_wrong_to_final_correct":106,"raw_correct_to_final_wrong":0
    },
    "GPT-OSS-120B":{
        "n":2000,"correct":1996,"semantic_correct":1715,"non_allow":1496,
        "unsafe_allow":0,"false_blocks":0,"raw_correct":1994,"raw_unsafe_allow":2,
        "raw_wrong_to_final_correct":2,"raw_correct_to_final_wrong":0
    }
}

LANG=pd.DataFrame([
    ["Qwen3.5-122B","English",0.962,1.000],
    ["Qwen3.5-122B","Romanized Hindi",0.902,0.996],
    ["Qwen3.5-122B","Romanized Telugu",0.626,0.998],
    ["Qwen3.5-122B","Telugu script",0.920,1.000],
    ["GPT-OSS-120B","English",0.880,1.000],
    ["GPT-OSS-120B","Romanized Hindi",0.872,0.992],
    ["GPT-OSS-120B","Romanized Telugu",0.802,1.000],
    ["GPT-OSS-120B","Telugu script",0.876,1.000],
],columns=["model","language","semantic_accuracy","decision_accuracy"])

def clopper_pearson(k,n,alpha=0.05):
    lo=0.0 if k==0 else beta.ppf(alpha/2,k,n-k+1)
    hi=1.0 if k==n else beta.ppf(1-alpha/2,k+1,n-k)
    return float(lo),float(hi)

def one_sided_zero_upper(n,alpha=0.05):
    return float(1-alpha**(1/n))

rows=[]
for model,d in PAPER.items():
    lo,hi=clopper_pearson(d["correct"],d["n"])
    rows.append({
        "model":model,"n":d["n"],"decision_accuracy":d["correct"]/d["n"],
        "accuracy_ci95_low":lo,"accuracy_ci95_high":hi,
        "semantic_accuracy":d["semantic_correct"]/d["n"],
        "observed_unsafe_allow_rate":d["unsafe_allow"]/d["non_allow"],
        "zero_unsafe_one_sided_95_upper":one_sided_zero_upper(d["non_allow"]),
        "raw_unsafe_allow":d["raw_unsafe_allow"]
    })
paper_stats=pd.DataFrame(rows)
display(paper_stats)

In [ ]:
tests=[]
for model,d in PAPER.items():
    b=d["raw_wrong_to_final_correct"]; c=d["raw_correct_to_final_wrong"]; n=b+c
    p=float(binomtest(min(b,c),n=n,p=0.5,alternative="two-sided").pvalue) if n else 1.0
    tests.append({"comparison":f"{model}: raw vs final","b":b,"c":c,"exact_p":p})

b,c=4,3
tests.append({
    "comparison":"Qwen final vs GPT-OSS final","b":b,"c":c,
    "exact_p":float(binomtest(min(b,c),n=b+c,p=0.5,alternative="two-sided").pvalue)
})
mcnemar=pd.DataFrame(tests)
display(mcnemar)

targets=[0.005,0.002,0.001,0.0005,0.0002,0.0001]
sample_size=pd.DataFrame([
    {"target_upper_rate":p,"required_zero_event_n":math.ceil(math.log(0.05)/math.log(1-p))}
    for p in targets
])
display(sample_size)

In [ ]:
dataset_path=NIYAM/"datasets/multilingual_agent_safety_dataset_10000.jsonl"
assert dataset_path.exists(),dataset_path
df=pd.read_json(dataset_path,lines=True)

def canonical_obj(x):
    return json.dumps(x,sort_keys=True,separators=(",",":"),default=str)

def norm_text(s):
    s=str(s).lower()
    s=re.sub(r"\d+","<NUM>",s)
    return re.sub(r"\s+"," ",s).strip()

df["_norm"]=df["raw_text"].map(norm_text)
audit={
    "n_rows":int(len(df)),
    "columns":list(df.columns),
    "exact_duplicate_raw_text":int(df.duplicated(subset=["raw_text"]).sum()),
    "template_normalized_duplicate_rows":int(df.duplicated(subset=["_norm"]).sum()),
    "unique_template_normalized_fraction":float(df["_norm"].nunique()/len(df))
}

def canon_slots(x):
    if isinstance(x,dict): obj=x
    elif x is None: obj={}
    else:
        try:
            if pd.isna(x): obj={}
            else: obj=json.loads(x) if isinstance(x,str) else x
        except Exception:
            obj={"raw":str(x)}
    return canonical_obj(obj)

if {"expected_intent","expected_slots"}.issubset(df.columns):
    df["_group_key"]=df["expected_intent"].astype(str)+"|"+df["expected_slots"].map(canon_slots)
    audit["semantic_groups"]=int(df["_group_key"].nunique())
    if "primary_lang" in df.columns:
        g=df.groupby("_group_key")["primary_lang"].nunique()
        audit["groups_spanning_multiple_languages"]=int((g>1).sum())

lang_col="primary_lang" if "primary_lang" in df.columns else ("language" if "language" in df.columns else None)
lang_audit=(df.groupby(lang_col).agg(
    n=("raw_text","size"),unique_raw=("raw_text","nunique"),unique_normalized=("_norm","nunique")
).reset_index() if lang_col else pd.DataFrame())

print(json.dumps(audit,indent=2))
display(lang_audit)

In [ ]:
sample=df.sample(min(2000,len(df)),random_state=20260911).copy()
vec=TfidfVectorizer(ngram_range=(1,2),min_df=2,max_features=12000)
X=vec.fit_transform(sample["_norm"])
sim=cosine_similarity(X)
np.fill_diagonal(sim,0.0)
nearest=sim.max(axis=1)
audit["near_duplicate_rate_sample_cosine_ge_0.95"]=float((nearest>=0.95).mean())
audit["near_duplicate_rate_sample_cosine_ge_0.90"]=float((nearest>=0.90).mean())
print("Near duplicate >=0.95:",audit["near_duplicate_rate_sample_cosine_ge_0.95"])

In [ ]:
LANG["gap"]=LANG["decision_accuracy"]-LANG["semantic_accuracy"]
display(LANG)

fig,ax=plt.subplots(figsize=(9,4.8))
x=np.arange(len(LANG))
ax.bar(x,LANG["gap"])
ax.set_xticks(x)
ax.set_xticklabels(LANG["model"]+"\n"+LANG["language"],rotation=45,ha="right")
ax.set_ylabel("Decision accuracy - semantic accuracy")
ax.set_title("Safety/semantic separation reported in the manuscript")
fig.tight_layout()
fig.savefig(RESULTS/"paper_semantic_decision_gap.png",dpi=220,bbox_inches="tight")
plt.show()

fig,ax=plt.subplots(figsize=(7,4.5))
ax.plot(sample_size["required_zero_event_n"],sample_size["target_upper_rate"],marker="o")
ax.set_xlabel("Zero-event non-Allow opportunities")
ax.set_ylabel("One-sided 95% upper risk bound")
ax.set_title("Sample size required to tighten zero-unsafe bound")
fig.tight_layout()
fig.savefig(RESULTS/"paper_zero_event_power.png",dpi=220,bbox_inches="tight")
plt.show()

In [ ]:
raw_dir=Path("/content/paper_raw")
raw_files=(sorted(raw_dir.glob("*.csv"))+sorted(raw_dir.glob("*.jsonl"))) if raw_dir.exists() else []

def load_any(p):
    return pd.read_csv(p) if p.suffix==".csv" else pd.read_json(p,lines=True)

def clustered_bootstrap_accuracy(d,B=5000,seed=42):
    required={"group_id","expected_verdict","final_verdict"}
    missing=required-set(d.columns)
    if missing:
        raise ValueError(f"missing columns: {sorted(missing)}")
    groups=d["group_id"].unique()
    rr=np.random.default_rng(seed)
    vals=[]
    for _ in range(B):
        sampled=rr.choice(groups,size=len(groups),replace=True)
        boot=pd.concat([d[d.group_id==g] for g in sampled],ignore_index=True)
        vals.append((boot.expected_verdict==boot.final_verdict).mean())
    return np.quantile(vals,[0.025,0.5,0.975]).tolist()

cluster_results={}
for p in raw_files:
    try:
        cluster_results[p.name]=clustered_bootstrap_accuracy(load_any(p))
    except Exception as e:
        cluster_results[p.name]={"error":str(e)}

if cluster_results:
    print(json.dumps(cluster_results,indent=2))
else:
    print("Exact per-case frozen files absent: no grouped confidence interval is fabricated.")

In [ ]:
required_artifacts={
    "frozen_holdout_with_group_id":False,
    "qwen_per_case_raw_and_final_results":False,
    "gpt_oss_per_case_raw_and_final_results":False,
    "exact_prompt_and_schema":False,
    "paper_SIL_IIEA_Anchor_runtime_source":False,
}
if raw_files:
    names=" ".join(p.name.lower() for p in raw_files)
    required_artifacts["frozen_holdout_with_group_id"]=any("holdout" in p.name.lower() for p in raw_files)
    required_artifacts["qwen_per_case_raw_and_final_results"]="qwen" in names
    required_artifacts["gpt_oss_per_case_raw_and_final_results"]=("gpt" in names or "oss" in names)

gap_lines=[
    "# NiyamTrace-X Reproducibility Gap Report","",
    "A missing item must not be described as publicly reproducible.",""
]
for k,v in required_artifacts.items():
    gap_lines.append(f"- [{'x' if v else ' '}] {k}")
gap_lines += [
    "","## Required before a 99%-readiness claim",
    "Publish the exact frozen holdout, per-case outputs, model revisions, prompts/schema,",
    "and the SIL/IIEA/Anchor/clarification runtime used for the manuscript tables.",
    "Then rerun this notebook with those files in /content/paper_raw/."
]
gap_text="\n".join(gap_lines)
print(gap_text)
(RESULTS/"REPRODUCIBILITY_GAP.md").write_text(gap_text)

In [ ]:
paper_stats.to_csv(RESULTS/"paper_aggregate_statistics.csv",index=False)
mcnemar.to_csv(RESULTS/"paper_mcnemar_tests.csv",index=False)
sample_size.to_csv(RESULTS/"zero_event_sample_size.csv",index=False)
LANG.to_csv(RESULTS/"paper_language_statistics.csv",index=False)
lang_audit.to_csv(RESULTS/"public_dataset_language_audit.csv",index=False)
(RESULTS/"public_dataset_audit.json").write_text(json.dumps(audit,indent=2))
(RESULTS/"clustered_bootstrap_if_available.json").write_text(json.dumps(cluster_results,indent=2))

tables=[
    "% Aggregate manuscript statistics\n"+paper_stats.to_latex(index=False,float_format=lambda x:f"{x:.6f}"),
    "% Exact paired tests\n"+mcnemar.to_latex(index=False,float_format=lambda x:f"{x:.6g}"),
    "% Public dataset language audit\n"+lang_audit.to_latex(index=False)
]
(RESULTS/"paper_tables.tex").write_text("\n\n".join(tables))

manifest=[]
for p in sorted(RESULTS.iterdir()):
    if p.is_file() and p.name!="SHA256SUMS.txt":
        manifest.append(f"{hashlib.sha256(p.read_bytes()).hexdigest()}  {p.name}")
(RESULTS/"SHA256SUMS.txt").write_text("\n".join(manifest)+"\n")

checks={
    "paper_counts_recomputed":len(paper_stats)==2,
    "paired_tests_recomputed":len(mcnemar)==3,
    "public_dataset_loaded":len(df)==10000,
    "no_grouped_ci_fabrication":bool(raw_files) or cluster_results=={},
}
print(json.dumps(checks,indent=2))
assert all(checks.values())

In [ ]:
import zipfile
zip_path=Path("/content/NTX_Q1_04_STATS_REPRO_RESULTS.zip")
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for p in RESULTS.iterdir():
        if p.is_file() and (
            p.name.startswith("paper_") or p.name.startswith("public_") or
            p.name.startswith("zero_") or p.name.startswith("clustered_") or
            p.name in {"REPRODUCIBILITY_GAP.md","SHA256SUMS.txt"}
        ):
            z.write(p,arcname=p.name)
print(zip_path)